In [3]:
!pip install geopy


In [1]:

addresses = [
    "avenue de l'Arsenal (grand rue), 17, Luxembourg",
    "avenue de l'Arsenal (Grand'Rue), 10, Luxembourg",
    "avenue de la Liberté, 16, Luxembourg",
    "avenue de la liberté, 26, Luxembourg",
    "avenue de la Liberté, 9, Luxembourg",
    "avenue Gaston Diderich, 2, Luxembourg",
    "avenue Monterey, 25, Luxembourg",
    "boulevard de la Pétrusse, 69, Luxembourg",
    "boulevard du Prince, 19, Luxembourg",
    "boulevard Joseph II, 28, Luxembourg",
    "boulevard Joseph II, 38, Luxembourg",
    "boulevard Prince Henri, 21, Luxembourg",
    "boulevard Royal 45, Luxembourg",
    "boulevard Royal, 21, Luxembourg",
    "boulevard Royal, 28, Luxembourg",
    "boulevard Royal, 2bis, Luxembourg",
    "boulevard Royal, 31, Luxembourg",
    "boulevard Royal, 33, Luxembourg",
    "boulevard royal, 45, Luxembourg",
    "Grand rue, 76, Luxembourg",
    "grand rue, 86, Luxembourg",
    "Pierret, 28, Luxembourg",
    "place de la gare, 80, Luxembourg",
    "route d'Itzig, 160, Luxembourg",
    "rue Aldringer, 14, Luxembourg",
    "rue Baudoin, 9, Luxembourg",
    "rue de Hollerich, 144, Luxembourg",
    "rue de Liberté, 22, Luxembourg",
    "rue Maréchal Foch, 15, Luxembourg",
    "rue Philippe II, 11, Luxembourg",
    "rue Philippe II, 5, Luxembourg",
    "route de Beggen, 269, Luxembourg"
]
from geopy.geocoders import Nominatim
import openai
import pandas as pd
import time
import urllib.parse

client = openai.OpenAI(api_key="sk-proj--.....................")  # <-- use your key


geolocator = Nominatim(user_agent="luxgeo-script")
results = []

for addr in addresses:
    lat, lon, osm_url, source, google_link, cleaned_address = "", "", "", "osm", "", ""
    print(f"Looking up: {addr}")
    try:
        location = geolocator.geocode(addr, timeout=5)
        if location:
            lat, lon = location.latitude, location.longitude
            osm_url = f"https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}"
            print(f"  → [OSM] {lat}, {lon}")
        else:
            raise Exception("Not found")
    except Exception:
        print(f"  → Not found with OSM. Try checking Google...")
        google_link = f"https://www.google.com/search?q={urllib.parse.quote(addr)}"
        print(f"    🔗 Check Google: {google_link}")
        
        # Optional: Prompt user to enter a corrected address
        cleaned_address = input("  ↪ Enter better address from Google result (or press Enter to skip): ").strip()
        if cleaned_address:
            try:
                location2 = geolocator.geocode(cleaned_address, timeout=5)
                if location2:
                    lat, lon = location2.latitude, location2.longitude
                    osm_url = f"https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}"
                    source = "osm_google_hint"
                    print(f"    → [OSM after Google hint] {lat}, {lon}")
                else:
                    raise Exception("Not found with improved address")
            except Exception:
                print("    → Still not found. Trying OpenAI...")
        
        if not lat:
            # If still not found, ask OpenAI
            try:
                chat_response = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[
                        {"role": "system", "content": "You are a geocoding assistant for addresses in Luxembourg. Give ONLY the latitude and longitude for each address as two numbers separated by a comma. No explanation, no text."},
                        {"role": "user", "content": f"Latitude and longitude for this Luxembourg address: {addr}?"}
                    ]
                )
                answer = chat_response.choices[0].message.content.strip()
                parts = answer.split(",")
                if len(parts) == 2:
                    lat, lon = parts[0].strip(), parts[1].strip()
                    osm_url = f"https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}"
                    source = "openai"
                    print(f"    → [OpenAI] {lat}, {lon}")
                else:
                    print(f"    → [OpenAI] Could not parse result: {answer}")
                    source = "not found"
            except Exception as ex:
                print(f"    → [OpenAI Error] {ex}")
                source = "not found"
    results.append([addr, cleaned_address, lat, lon, osm_url, source, google_link])
    time.sleep(1)

# Write to CSV
df = pd.DataFrame(results, columns=["original_address", "google_hint_address", "latitude", "longitude", "osm_link", "source", "google_search_link"])
df.to_csv("lux_addresses_geocoded_humanloop.csv", sep='\t', encoding='utf-8')
print("✅ Done! Check lux_addresses_geocoded_humanloop.csv")


Looking up: avenue de l'Arsenal (grand rue), 17, Luxembourg
  → Not found with OSM. Try checking Google...
    🔗 Check Google: https://www.google.com/search?q=avenue%20de%20l%27Arsenal%20%28grand%20rue%29%2C%2017%2C%20Luxembourg


  ↪ Enter better address from Google result (or press Enter to skip):  Arsenalavenue 17 avenue de l'Arsenal, Stad - Luxemburg


    → Still not found. Trying OpenAI...
    → [OpenAI] 49.6112, 6.1239
Looking up: avenue de l'Arsenal (Grand'Rue), 10, Luxembourg
  → Not found with OSM. Try checking Google...
    🔗 Check Google: https://www.google.com/search?q=avenue%20de%20l%27Arsenal%20%28Grand%27Rue%29%2C%2010%2C%20Luxembourg


  ↪ Enter better address from Google result (or press Enter to skip):  Arsenalavenue 10 avenue de l'Arsenal, Stad - Luxemburg


    → Still not found. Trying OpenAI...
    → [OpenAI] 49.6006, 6.1305
Looking up: avenue de la Liberté, 16, Luxembourg
  → [OSM] 49.6058845, 6.1287571
Looking up: avenue de la liberté, 26, Luxembourg
  → [OSM] 49.6044423, 6.1298776
Looking up: avenue de la Liberté, 9, Luxembourg
  → [OSM] 49.6067334, 6.1287018
Looking up: avenue Gaston Diderich, 2, Luxembourg
  → [OSM] 49.6101368, 6.1186141
Looking up: avenue Monterey, 25, Luxembourg
  → [OSM] 49.6104617, 6.1248505
Looking up: boulevard de la Pétrusse, 69, Luxembourg
  → [OSM] 49.6067541, 6.1260863
Looking up: boulevard du Prince, 19, Luxembourg
  → Not found with OSM. Try checking Google...
    🔗 Check Google: https://www.google.com/search?q=boulevard%20du%20Prince%2C%2019%2C%20Luxembourg


...

    → Still not found. Trying OpenAI...
    → [OpenAI] 49.603534, 6.134898
Looking up: boulevard Joseph II, 28, Luxembourg
  → [OSM] 49.613364, 6.1214477
Looking up: boulevard Joseph II, 38, Luxembourg
  → [OSM] 49.6139977, 6.1218343
Looking up: boulevard Prince Henri, 21, Luxembourg
  → [OSM] 49.4906222, 5.9766912
Looking up: boulevard Royal 45, Luxembourg
  → [OSM] 49.6135694, 6.1268755
Looking up: boulevard Royal, 21, Luxembourg
  → [OSM] 49.6134663, 6.1273751
Looking up: boulevard Royal, 28, Luxembourg
  → [OSM] 49.6135694, 6.1268755
Looking up: boulevard Royal, 2bis, Luxembourg
  → [OSM] 49.6135694, 6.1268755
Looking up: boulevard Royal, 31, Luxembourg
  → [OSM] 49.6135694, 6.1268755
Looking up: boulevard Royal, 33, Luxembourg
  → [OSM] 49.6135694, 6.1268755
Looking up: boulevard royal, 45, Luxembourg
  → [OSM] 49.6135694, 6.1268755
Looking up: Grand rue, 76, Luxembourg
  → [OSM] 49.6124295, 6.127513
Looking up: grand rue, 86, Luxembourg
  → [OSM] 49.612329, 6.1270109
Looking up: 

  ↪ Enter better address from Google result (or press Enter to skip):  9 Rue Baudouin, luxembourg


    → [OSM after Google hint] 49.5978381, 6.1207514
Looking up: rue de Hollerich, 144, Luxembourg
  → [OSM] 49.5986956, 6.1255988
Looking up: rue de Liberté, 22, Luxembourg
  → [OSM] 49.5549264, 5.8752473
Looking up: rue Maréchal Foch, 15, Luxembourg
  → [OSM] 49.6084111, 6.1104601
Looking up: rue Philippe II, 11, Luxembourg
  → [OSM] 49.6117411, 6.1280809
Looking up: rue Philippe II, 5, Luxembourg
  → [OSM] 49.6117411, 6.1280809
Looking up: route de Beggen, 269, Luxembourg
  → Not found with OSM. Try checking Google...
    🔗 Check Google: https://www.google.com/search?q=route%20de%20Beggen%2C%20269%2C%20Luxembourg


  ↪ Enter better address from Google result (or press Enter to skip):  269 Rue de Beggen


    → [OSM after Google hint] 49.6475771, 6.127658
✅ Done! Check lux_addresses_geocoded_humanloop.xlsx


In [ ]:
!pip install folium

In [1]:
import folium
from folium.plugins import HeatMap
import pandas as pd

In [2]:
df = pd.read_csv('addresses01.csv', delimiter=',')

In [3]:
addresses = list(df['nom de la rue'])
firstHalf = list(df['1929-1939'])
secondHalf = list(df['1945-1959'])
latitudes = list(df['latitude'])
longitudes = list(df['longitude'])

In [4]:
geoFirst = []
geoSecond = []
for i in range(len(latitudes)):
    if firstHalf[i] != 0:
        for j in range(firstHalf[i]):
            geoFirst.append([latitudes[i], longitudes[i]])
    if secondHalf[i] != 0:
        for k in range(secondHalf[i]):
            geoSecond.append([latitudes[i], longitudes[i]])

In [5]:
# Create maps centered on Luxembourg City
m1 = folium.Map(
    location=[49.6116, 6.1319],  # Luxembourg City coordinates
    zoom_start=12.5,  # Adjust zoom level (higher = more zoomed in)
    tiles="OpenStreetMap"  # Default map style (can be changed)
)

m2 = folium.Map(
    location=[49.6116, 6.1319],  # Luxembourg City coordinates
    zoom_start=12.5,  # Adjust zoom level (higher = more zoomed in)
    tiles="OpenStreetMap"  # Default map style (can be changed)
)

In [6]:
HeatMap(geoFirst).add_to(m1)
HeatMap(geoSecond).add_to(m2)

In [7]:
display(m1)
m1.save("heatmap_1929-1940.html")

In [8]:
display(m2)
m1.save("heatmap_1945-1960.html")

In [ ]:
m0 = folium.Map(
    location=[49.6116, 6.1319],  # Luxembourg City coordinates
    zoom_start=12.5,  # Adjust zoom level (higher = more zoomed in)
    tiles="OpenStreetMap"  # Default map style (can be changed)
)

In [ ]:
# Heatmap 1 (1929-1940)
heatmap1 = HeatMap(
    data=geoFirst,
    name="1929-1940",  # Name for LayerControl
    radius=15,
    gradient={0.4: "blue", 0.6: "lime", 1: "red"},  # Custom colors
).add_to(m0)

# Heatmap 2 (1945-1960)
heatmap2 = HeatMap(
    data=geoSecond,
    name="1945-1960",  # Name for LayerControl
    radius=15,
    gradient={0.4: "purple", 0.6: "orange", 1: "darkred"},
).add_to(m0)

In [ ]:
folium.LayerControl(collapsed=False).add_to(m0)

In [ ]:
display(m0)